In [1]:
from src.config import OllamaConfig

OLLAMA_HOST = "172.19.176.1"
OLLAMA_PORT = 11434
OLLAMA_URL = f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/chat"

config = OllamaConfig(embedding_model="BAAI/bge-base-en", ollama_url=OLLAMA_URL)

print("Base directory:", config.base_dir)

config.ensure_dirs()

Base directory: /mnt/c/dev/ml/rag-qa
✅ Ensured directory exists: /mnt/c/dev/ml/rag-qa/.hf_cache
✅ Ensured directory exists: /mnt/c/dev/ml/rag-qa/data
✅ Ensured directory exists: /mnt/c/dev/ml/rag-qa/data/train
✅ Ensured directory exists: /mnt/c/dev/ml/rag-qa/data/validation
✅ Ensured directory exists: /mnt/c/dev/ml/rag-qa/data/test


In [ ]:
from src.data_loader import DataLoader

data_loader = DataLoader(config=config)
corpus, corpus_embeddings = data_loader.load_existing_embeddings()

🔹 Loaded FAISS index with 978526 passages


In [4]:
from src.retriever import Retriever
from src.reranker import Reranker
from src.generator import OllamaGenerator

retriever = Retriever(config=config)
reranker = Reranker(config=config)
generator = OllamaGenerator(config=config)

In [7]:
query = "What is the capital of France?"
candidates = 100
top_k = 3

retrieved_passages, _, candidates_idx = retriever.retrieve(
    query=query, 
    corpus=corpus,
    faiss_index=corpus_embeddings, 
    top_k=candidates)

context, _ = reranker.rerank_and_get_top_k(
    query=query, 
    corpus=corpus, 
    candidates_idx=candidates_idx,
    top_k=top_k)

answer = generator.generate(query=query, passages=context)

print("\nGenerated Answer:\n", answer)


Generated Answer:
 Paris.


In [6]:
print("\n🔍 Used Context Passages:\n")
for i,p in enumerate(context, 1):
    print(f"{i}. {p[:200].replace(chr(10),' ')}...\n")


🔍 Used Context Passages:

1. Paris: paris ( french : ) is the capital and most populous city of france. situated on the river seine in northern metropolitan france, it is in the centre of the ile - de - france region, also known ...

2. France: france ( french : ), officially the french republic ( ), is a sovereign state comprising territory in western europe and several overseas regions and territories. the european, or metropolitan...

3. Capital city: valparaiso. * : prague is the sole constitutional capital. brno is home to all three of the country's highest courts, making it the de facto capital of the czech judicial branch. * : the...

